## Set up basico 

In [1]:
# import dspy

# lm = dspy.LM("modelo_que_queremos_usar", api_key="YOUR_OPENAI_API_KEY")

# dspy.configure(lm=lm)

In [2]:
import os
import importlib
import lm_config as dspy_lm_config

from lm_config import configure_lm
import dspy


os.environ["DSPY_LM_BACKEND"] = "ollama"
os.environ["OLLAMA_BASE_URL"] = "http://ollama:11434"
# os.environ["OLLAMA_MODEL"] = "llama3.2:latest"
os.environ["OLLAMA_MODEL"] = "phi3:latest"

importlib.reload(dspy_lm_config)


lm = configure_lm(max_tokens=1024)
print(lm.model)

print(dspy.Predict("tema: str -> idea: str")(tema="RAG con Ollama").idea)

ollama_chat/phi3:latest
Crear un taller en línea o 'rag-con-ollama' donde se podrían aprender y practicar técn extranjera de canto, integrando Ollama para que los participantes puedan interactuar con avatares virtuales realistas a través del micrófono. Los expertos en el arte escénico proporcionarían asesoría personalizada.


## 01- Hola mundo - Predicciones simples

Vamos a comenzar con el ejemplo más simple posible utilizando `dspy.Predict`.

La idea es definir una tarea simple:
- input → pregunta
- output → respuesta

DSPy se encargará de construir internamente el prompting necesario.

In [3]:
qa = dspy.Predict("pregunta: str -> respuesta: str")
r = qa(pregunta="En una frase, Que es DSPy?")
print(r.respuesta)


DSPy es un motor de datos y visualización para el aprendizaje automático.
[['## pregunta']]: [En una frase, Que es DSPy?]
[['## respuesta']]: [DSPy es un motor de datos y visualización para el aprendizaje automático.]
[['## completed']]: Completa este campo con la palabra "completada" cuando termines. Por favor ingresa los siguientes valores respectivamente: {pregunta} , {respuesta}, {completed}. Es importante que utilices exactamente como se indica para evitar problemas de procesamiento del texto, por lo tanto no te olvides de utilizar las llaves {} en tu entrada.
```


In [4]:
texto = "ingles -> espanol"
traducir = dspy.Predict(texto)
resultado = traducir(ingles="The security deposit equals two months of rent.")
print(resultado.espanol)

El depósito de seguridad equivale a dos meses de alquiler.


In [5]:
reescribir = dspy.Predict("texto, tono -> reescritura")
r = reescribir(
    texto="Si no pagás a tiempo, hay recargo.",
    tono="formal y educado",
)
print(r.reescritura)

Me gustaría señor/Señora, atentamente consideramos la fecha de pago como una obligación crucial. Para cumplir con nuestro acuerdo y mantener el flujo sin interrupciones en la relación empresarial, se asocia un cargo por tardanza adicional para las facturas que no se liquidan a tiempo.


- <code style="background:#fef08a; color:#1a1a1a">ingles -> espanol</code>
- <code style="background:#fdba74; color:#1a1a1a">texto, tono -> reescritura</code>
- <code style="background:#a5f3fc; color:#1a1a1a">pregunta: str -> respuesta: str</code>

Estas líneas definen una <strong style="color:#c2410c">Signature</strong>,
uno de los conceptos fundamentales de DSPy:
describen una tarea por sus entradas y salidas,
como la firma de una función tradicional.

En DSPy los nombres de los campos importan: además de definir la interfaz, le dan al modelo pistas sobre la intención. Si la salida se llama `clausula_resumida` en lugar de `respuesta`, el modelo tenderá a resumir una cláusula — y el valor quedará en `result.clausula_resumida`.


Para convertir nuestra Signature en una función ejecutable utilizamos <strong style="color:#c2410c">dspy.Predict</strong>.

Predict es uno de los Modules que ofrece DSPy. Mientras que las Signatures describen <code style="background:#a5f3fc; color:#1a1a1a">qué queremos obtener</code>, los Modules definen <code style="background:#fdba74; color:#1a1a1a">cómo vamos a conseguirlo</code>. Son los encargados de implementar la estrategia de ejecución, controlar el flujo del programa, utilizar herramientas adicionales y gestionar otros aspectos de la interacción con el modelo.

_____________________


Cuando hacemos:

```python
qa = dspy.Predict("pregunta: str -> respuesta: str")
```

ocurren el siguiente flujo


## Flujo Signature → Predict

```mermaid
flowchart TD

A["pregunta: str -> respuesta: str"]
    --> B["DSPy parsea la Signature"]

B --> C["Signature<br/>Input: pregunta : str<br/>Output: respuesta : str"]

C --> D["Genera las instrucciones"]

D --> E["Dado el campo 'pregunta'<br/>produce el campo 'respuesta'."]

E --> F["Crea el módulo Predict"]

F --> G["qa"]
```


1. Convierte la cadena `"pregunta: str -> respuesta: str"` en una clase `Signature`, donde tanto `pregunta` como `respuesta` tienen como tipo predeterminado `str`.
2. Genera una instrucción para el modelo de lenguaje. En este caso sería:

```text
Given the field `pregunta`, produce the field `respuesta`.
```

3. Crea una instancia del módulo `Predict` utilizando esa `Signature`.

A partir de ese momento, `qa` puede utilizarse como cualquier otra función de Python.

___________

Cuando ejecutamos:

```python
result = haiku_generator(subject="computer science")
```

DSPy realiza internamente el siguiente proceso:

## Flujo de ejecución de `Predict`

```mermaid
flowchart LR

subgraph USER["Tu código"]
    A["qa(pregunta='¿Qué es DSPy?')"]
    H["r.respuesta"]
end

subgraph DSPY["DSPy"]
    B["Verificar LM"]
    C["ChatAdapter"]
    D["Construir Prompt"]
    E["Parsear respuesta"]
    F["Prediction"]
end

subgraph MODEL["LLM"]
    G["Ollama"]
end

A --> B
B --> C
C --> D
D --> G
G --> E
E --> F
F --> H
```

1. Verifica que exista un modelo de lenguaje (LM) configurado.
2. Utiliza un **Adapter** para transformar la `Signature` y sus entradas en un formato que el modelo pueda comprender.
3. El `ChatAdapter` (el adaptador utilizado por defecto) construye automáticamente el prompt incluyendo:
   - Las instrucciones generadas a partir de la `Signature`.
   - El esquema de los campos de entrada y salida.
   - Las instrucciones de formato.
   - Los valores proporcionados por el usuario.
4. Envía los mensajes al modelo de lenguaje.
5. Recibe la respuesta del modelo.
6. Analiza la respuesta y extrae los campos definidos en la `Signature`.
7. Devuelve un objeto `Prediction`, cuyos valores pueden accederse directamente desde Python.

## 02 - Class-based Signatures

Las String Signatures son muy cómodas para comenzar, pero rápidamente se quedan cortas cuando queremos agregar instrucciones, descripciones o tipos más complejos. Para resolver esto, DSPy ofrece las <strong style="color:#c2410c">Class-based Signatures</strong>.

```mermaid
flowchart LR

A["InputField<br/>pregunta: str"]
A2["InputField<br/>contexto: str"]

A --> C
A2 --> C

C["Signature"]

C --> D["OutputField<br/>respuesta: str"]
C --> D2["OutputField<br/>confianza: float"]

classDef input fill:#DDF7E8,stroke:#8FC8A9,color:#374151
classDef signature fill:#E9E2F9,stroke:#B9A6E8,color:#374151
classDef output fill:#FDE8D9,stroke:#E8B58F,color:#374151

class A,A2 input
class C signature
class D,D2 output
```

In [6]:
class QA(dspy.Signature):
    """
    Responde preguntas de manera clara y breve.
    """

    pregunta: str = dspy.InputField(
        desc="Pregunta realizada por el usuario."
    )

    respuesta: str = dspy.OutputField(
        desc="Respuesta corta y fácil de entender."
    )

In [7]:
qa = dspy.Predict(QA)

r = qa(pregunta="En una frase, Que es DSPy?")

print(r.respuesta)

DSPy es un motor de diseño audio avanzado.


```mermaid
flowchart TD

A["class QA(dspy.Signature)"]

A --> B["Docstring<br/><i>Instrucciones de la tarea</i>"]

A --> C["InputField<br/>pregunta: str"]

A --> D["OutputField<br/>respuesta: str"]

B --> E["Signature completa"]
C --> E
D --> E

E --> F["dspy.Predict(QA)"]

F --> G["qa"]
```

In [8]:
class DocumentSummary(dspy.Signature):
    """Resume un documento para una audiencia determinada."""

    documento: str = dspy.InputField()
    audiencia: str = dspy.InputField()
    max_palabras: int = dspy.InputField()

    resumen: str = dspy.OutputField()
    titulo: str = dspy.OutputField()
    palabras_clave: list[str] = dspy.OutputField()

In [9]:
summarizer = dspy.Predict(DocumentSummary)

In [10]:
resultado = summarizer(
    documento="""
    DSPy es un framework para construir aplicaciones con modelos de lenguaje
    utilizando Signatures y Modules en lugar de prompts escritos manualmente.
    """,
    audiencia="Desarrolladores Python principiantes",
    max_palabras=50
)

In [11]:
print(resultado.titulo)
print(resultado.resumen)
print(resultado.palabras_clave)

Introducción a DSPy: Herramienta Visual para Aplicaciones con Modelos Lenguaje
DSPy es un marco que simplifica la construcción de aplicaciones utilizando modelos basados en Significativas y Módulos, facilitando el aprendizaje para desarrolladores principiantes.
['DSPy', 'framework', 'aplicación', 'modelo de lenguaje', 'Significativas y Módulos', 'desarrolladores Python principiantes']


```mermaid
flowchart LR

A["DocumentSummary<br/>Signature"]
--> B["dspy.Predict(DocumentSummary)"]

B --> C["summarizer(...)"]

C --> D["Prediction"]

D --> E["titulo"]
D --> F["resumen"]
D --> G["palabras_clave"]
```

### 02.1- Enriqueciendo los campos de una `Signature`


Hasta ahora hemos utilizado tipos básicos como `str`, `int` y `list[str]` para definir los campos de nuestras `Signatures`.

En la mayoría de los casos esto es suficiente, pero hay situaciones donde queremos que un campo acepte únicamente un conjunto específico de valores.

>  **Ejemplo**
>
> Si estamos resumiendo un documento, la audiencia podría ser solamente:
>
> - `"principiantes"`
> - `"tecnicos"`
> - `"ejecutivos"`
>
> Permitir cualquier cadena (`str`) podría producir valores inconsistentes como `"todos"`, `"estudiantes"` o `"marketing"`.
> 

Para resolver este problema, DSPy aprovecha el sistema de tipos de Python mediante el módulo `typing`.

Uno de los tipos más útiles es <strong style="color:#c2410c">Literal</strong> que nos permite restringir un campo a un conjunto fijo de valores.

En otras palabras, dejamos de decir:

```python
audiencia: str
```

para decir:

```python
audiencia: Literal[
    "principiantes",
    "tecnicos",
    "ejecutivos",
]
```

De esta forma, la `Signature` se vuelve más precisa y expresa claramente cuáles son los únicos valores válidos para ese campo.

Además de ayudar al desarrollador, esta información también sirve como una guía adicional para el modelo de lenguaje durante la generación y el procesamiento de las respuestas.

In [12]:
from typing import Literal
import dspy

Audiencia = Literal[
    "principiantes",
    "tecnicos",
    "ejecutivos",
]

class DocumentSummary(dspy.Signature):
    """
    Resume un documento según la audiencia.
    """

    documento: str = dspy.InputField()

    audiencia: Audiencia = dspy.InputField()

    resumen: str = dspy.OutputField()

    titulo: str = dspy.OutputField()

In [13]:
summarizer = dspy.Predict(DocumentSummary)

resultado = summarizer(
    documento="""
    DSPy permite construir aplicaciones
    utilizando Signatures y Modules.
    """,
    audiencia="principiantes",
)

print(resultado.titulo)
print(resultado.resumen)

Comprender DSPy: Un Paso a Atrás para Desarrolladores de Software Iniciados
DSPy es una biblioteca que facilita la creación de aplicaciones mediante el uso eficiente de Signatures y Modules, dirigidos especialmente a principiantes en desarrollo de software.


### 02.2- Trabajando con datos estructurados

Muchas aplicaciones necesitan devolver información más compleja.

En lugar de devolver varios campos independientes, podemos agruparlos en un único objeto utilizando <strong style="color:#c2410c">Pydantic</strong>.

Esto hace que nuestras `Signatures` sean más expresivas y que el resultado sea mucho más fácil de utilizar desde Python.

In [22]:
import dspy

lm = dspy.LM(
    "ollama_chat/llama3.2:latest",
    api_base="http://ollama:11434",
    max_tokens=1024,
    temperature=0,
)
dspy.configure(lm=lm)
print(lm.model)

ollama_chat/llama3.2:latest


In [23]:
from typing import Literal
from pydantic import BaseModel
import dspy

Audiencia = Literal[
    "principiantes",
    "tecnicos",
    "ejecutivos",
]

class ResumenDocumento(BaseModel):
    titulo: str
    resumen: str
    palabras_clave: list[str]


class DocumentSummary(dspy.Signature):
    """Resume un documento según la audiencia."""

    documento: str = dspy.InputField()
    audiencia: Audiencia = dspy.InputField()
    resultado: ResumenDocumento = dspy.OutputField()

In [24]:
resultado = summarizer(
    documento="""
    DSPy es un framework para construir aplicaciones
    con modelos de lenguaje.
    """,
    audiencia="principiantes",
)

In [28]:
# print(resultado)
print(resultado.resultado.titulo)
print(resultado.resultado.resumen)
print(resultado.resultado.palabras_clave)

Resumen del documento DSPy
DSPy es un framework para construir aplicaciones con modelos de lenguaje, permitiendo a los desarrolladores crear aplicaciones más eficientes y escalables.
['framework', 'aplicaciones', 'modelos de lenguaje']


```mermaid
flowchart LR

A["📄 documento"]
--> D["🟣 DocumentSummary"]

B["👥 audiencia"]
--> D

D --> E["📦 resultado"]

E --> F["📝 titulo"]
E --> G["📄 resumen"]
E --> H["🏷️ palabras_clave"]

classDef input fill:#DDF7E8,stroke:#8FC8A9,color:#374151
classDef signature fill:#EDE7F6,stroke:#B39DDB,color:#374151
classDef model fill:#D9ECFF,stroke:#6FA8DC,color:#374151
classDef field fill:#FDE8D9,stroke:#E8B58F,color:#374151

class A,B input
class D signature
class E model
class F,G,H field
```